# Sifter Reviewed Reranker Training

Run this notebook in Google Colab from top to bottom.

What it does:
1. Installs training packages.
2. Uploads the reviewed training zip.
3. Trains the reranker on human-reviewed labels.
4. Prints metrics.
5. Uploads the model to Hugging Face.

Before starting, set Colab runtime to GPU:
`Runtime -> Change runtime type -> GPU`

## 1. Settings

Fill these once, then run every cell below.

In [ ]:
#@title Training Settings

HF_MODEL_ID = "shikharshahi/sifter-redrob-reranker"  #@param {type:"string"}
BASE_MODEL = "distilbert-base-uncased"  #@param {type:"string"}

# Keep this blank. The notebook will securely ask for your token later.
HF_TOKEN = ""  #@param {type:"string"}

EPOCHS = 3  #@param {type:"number"}
BATCH_SIZE = 8  #@param {type:"integer"}
LEARNING_RATE = 2e-5  #@param {type:"number"}
MAX_LENGTH = 256  #@param {type:"integer"}

DATA_ZIP_NAME = "redrob-reranker-reviewed.zip"
DATA_DIR = "/content/data/redrob-reranker-reviewed"
OUTPUT_DIR = "/content/outputs/sifter-redrob-reranker-reviewed"


## 2. Install Packages

In [ ]:
!pip install -q "transformers>=4.40.0" "datasets>=2.18.0" "accelerate>=0.28.0" scikit-learn scipy huggingface_hub


## 3. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. Training will still work, but it may be slow. In Colab, go to Runtime -> Change runtime type -> GPU.")


## 4. Upload Reviewed Training Data

Upload this file from your machine:

`data/redrob-reranker-reviewed.zip`

In [ ]:
from google.colab import files
from pathlib import Path
import shutil
import zipfile

zip_path = Path("/content") / DATA_ZIP_NAME

if not zip_path.exists():
    print(f"Upload {DATA_ZIP_NAME} now...")
    uploaded = files.upload()
    if DATA_ZIP_NAME not in uploaded:
        uploaded_name = next(iter(uploaded.keys()))
        shutil.move(f"/content/{uploaded_name}", zip_path)

extract_root = Path("/content/extracted_review_data")
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_root)

data_dir = Path(DATA_DIR)
data_dir.mkdir(parents=True, exist_ok=True)

# The zip may contain files at root, inside redrob-reranker-reviewed/, or inside data/redrob-reranker-reviewed/.
train_matches = list(extract_root.rglob("reranker_train.jsonl"))
valid_matches = list(extract_root.rglob("reranker_valid.jsonl"))
summary_matches = list(extract_root.rglob("summary.json"))

assert train_matches, "Missing reranker_train.jsonl inside uploaded zip"
assert valid_matches, "Missing reranker_valid.jsonl inside uploaded zip"

shutil.copy2(train_matches[0], data_dir / "reranker_train.jsonl")
shutil.copy2(valid_matches[0], data_dir / "reranker_valid.jsonl")
if summary_matches:
    shutil.copy2(summary_matches[0], data_dir / "summary.json")

print("Data folder:", data_dir)
print("Files:", [p.name for p in data_dir.iterdir()])
assert (data_dir / "reranker_train.jsonl").exists(), "Missing reranker_train.jsonl"
assert (data_dir / "reranker_valid.jsonl").exists(), "Missing reranker_valid.jsonl"


## 5. Login To Hugging Face

Use a Hugging Face **write** token. If `HF_TOKEN` above is blank, this cell asks for it securely.

In [ ]:
import getpass
from huggingface_hub import login

token = HF_TOKEN.strip() or getpass.getpass("Paste Hugging Face WRITE token: ")
login(token=token, add_to_git_credential=False)
print("Logged in. Model will upload to:", HF_MODEL_ID)


## 6. Load Dataset And Tokenizer

In [ ]:
import json
import numpy as np
from datasets import load_dataset
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error
from transformers import AutoTokenizer, DataCollatorWithPadding

dataset = load_dataset(
    "json",
    data_files={
        "train": str(Path(DATA_DIR) / "reranker_train.jsonl"),
        "validation": str(Path(DATA_DIR) / "reranker_valid.jsonl"),
    },
)

print("Train rows:", len(dataset["train"]))
print("Validation rows:", len(dataset["validation"]))
print("Example:", dataset["train"][0]["candidate_id"], dataset["train"][0]["label"])

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

def tokenize(batch):
    pairs = [
        f"Job description:\n{query}\n\nCandidate profile:\n{text}"
        for query, text in zip(batch["query"], batch["text"])
    ]
    encoded = tokenizer(pairs, truncation=True, max_length=MAX_LENGTH)
    encoded["labels"] = [float(value) for value in batch["label"]]
    return encoded

tokenized = dataset.map(tokenize, batched=True, remove_columns=dataset["train"].column_names)


## 7. Create Model And Trainer

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=1,
    problem_type="regression",
    ignore_mismatched_sizes=True,
)

class RegressionTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels").float()
        outputs = model(**inputs)
        scores = outputs.logits.reshape(-1).float()
        loss = torch.nn.functional.mse_loss(scores, labels.reshape(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.asarray(predictions).reshape(-1)
    labels = np.asarray(labels).reshape(-1)
    spearman = spearmanr(labels, preds).correlation
    if np.isnan(spearman):
        spearman = 0.0
    return {
        "rmse": float(np.sqrt(mean_squared_error(labels, preds))),
        "mae": float(mean_absolute_error(labels, preds)),
        "spearman": float(spearman),
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="spearman",
    greater_is_better=True,
    fp16=False,
    bf16=False,
    report_to="none",
    push_to_hub=True,
    hub_model_id=HF_MODEL_ID,
)

trainer = RegressionTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

print("Trainer ready.")


## 8. Train

In [ ]:
trainer.train()


## 9. Evaluate And Save

In [ ]:
metrics = trainer.evaluate()
print(json.dumps(metrics, indent=2))

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
Path(OUTPUT_DIR, "eval_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print("Saved locally to:", OUTPUT_DIR)


## 10. Upload To Hugging Face

In [ ]:
trainer.push_to_hub()
print("Uploaded model:", f"https://huggingface.co/{HF_MODEL_ID}")
